In [7]:
import torch
import torch.nn as nn
import torch.onnx as onnx

In [8]:
model_path = "../simple_conv.onnx"

In [9]:
# A small CNN exercising every layer type currently supported by the Rust runtime:
# Conv2d -> ReLU -> Conv2d -> ReLU -> Flatten -> Linear -> Sigmoid -> Linear -> Tanh -> Linear -> Softmax
#
# Input is (1, 1, 4, 4): batch size 1, 1 channel, 4x4 spatial. Both convs use
# kernel_size=3, stride=1, padding=1 so spatial dims stay 4x4 throughout ("same" padding),
# which keeps the flattened size (8 * 4 * 4 = 128) easy to check by hand.


class SimpleConvModel(nn.Module):
    def __init__(self):
        super(SimpleConvModel, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, stride=1, padding=1)
        self.act_1_relu = nn.ReLU()
        self.conv2 = nn.Conv2d(in_channels=4, out_channels=8, kernel_size=3, stride=1, padding=1)
        self.act_2_relu = nn.ReLU()
        self.flatten = nn.Flatten()
        self.linear1 = nn.Linear(8 * 4 * 4, 20)
        self.act_3_sigmoid = nn.Sigmoid()
        self.linear2 = nn.Linear(20, 15)
        self.act_4_tanh = nn.Tanh()
        self.linear3 = nn.Linear(15, 5)
        self.act_5_softmax = nn.Softmax(dim=1)  # Apply softmax along the feature dimension

    def forward(self, x):
        output = self.conv1(x)
        output = self.act_1_relu(output)
        output = self.conv2(output)
        output = self.act_2_relu(output)
        output = self.flatten(output)
        output = self.linear1(output)
        output = self.act_3_sigmoid(output)
        output = self.linear2(output)
        output = self.act_4_tanh(output)
        output = self.linear3(output)
        output = self.act_5_softmax(output)
        return output


# Example usage
model = SimpleConvModel()
print(model)

print("Model weights:")
for name, param in model.named_parameters():
    print(f"{name}: {param.shape}")
    if "weight" in name:
        print(f"  Weight values (first 5): {param.flatten()[:5]}")
    elif "bias" in name:
        print(f"  Bias values (first 5): {param.flatten()[:5]}")

SimpleConvModel(
  (conv1): Conv2d(1, 4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (act_1_relu): ReLU()
  (conv2): Conv2d(4, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (act_2_relu): ReLU()
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear1): Linear(in_features=128, out_features=20, bias=True)
  (act_3_sigmoid): Sigmoid()
  (linear2): Linear(in_features=20, out_features=15, bias=True)
  (act_4_tanh): Tanh()
  (linear3): Linear(in_features=15, out_features=5, bias=True)
  (act_5_softmax): Softmax(dim=1)
)
Model weights:
conv1.weight: torch.Size([4, 1, 3, 3])
  Weight values (first 5): tensor([-0.2848,  0.0165,  0.2352, -0.0161,  0.1030], grad_fn=<SliceBackward0>)
conv1.bias: torch.Size([4])
  Bias values (first 5): tensor([-0.3160,  0.2267,  0.1080, -0.0417], grad_fn=<SliceBackward0>)
conv2.weight: torch.Size([8, 4, 3, 3])
  Weight values (first 5): tensor([ 0.0054, -0.1491,  0.0907, -0.1537, -0.0981], grad_fn=<SliceBackward0>)
conv2.bias: torch.Size([8])
  

In [10]:
# export to onnx
dummy_input = torch.randn(1, 1, 4, 4)
onnx.export(model, dummy_input, model_path, export_params=True, opset_version=11)

/tmp/ipykernel_34386/3045890397.py:3: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  onnx.export(model, dummy_input, model_path, export_params=True, opset_version=11)
W0815 18:06:25.934000 34386 site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `SimpleConvModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SimpleConvModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/home/david/miniconda3/envs/pytorch/lib/python3.14/copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 11).
Failed to convert the model to the target version 11 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/home/david/miniconda3/envs/pytorch/lib/python3.14/site-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/home/david/miniconda3/envs/pytorch/lib/python3.14/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/home/david/miniconda3/envs/pytor

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.13.0+cu132',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"x"<FLOAT,[1,1,4,4]>
            ),
            outputs=(
                %"softmax"<FLOAT,[1,5]>
            ),
            initializers=(
                %"conv1.weight"<FLOAT,[4,1,3,3]>{TorchTensor(...)},
                %"conv1.bias"<FLOAT,[4]>{TorchTensor<FLOAT,[4]>(Parameter containing: tensor([-0.3160,  0.2267,  0.1080, -0.0417], requires_grad=True), name='conv1.bias')},
                %"conv2.weight"<FLOAT,[8,4,3,3]>{TorchTensor(...)},
                %"conv2.bias"<FLOAT,[8]>{TorchTensor<FLOAT,[8]>(Parameter containing: tensor([ 0.1023,  0.0386, -0.0475, -0.0907,  0.0602,  0.1536, -0.0441,  0.0299], requires_grad=True), name='conv2.bias')},
              

In [11]:
# run the model with pytorch
# 4x4 single-channel "image" with values 1..16, so it's easy to eyeball against the Rust output
input_data = torch.arange(1, 17, dtype=torch.float32).reshape(1, 1, 4, 4)
with torch.no_grad():
    output = model(input_data)
print(input_data)
print(output)

tensor([[[[ 1.,  2.,  3.,  4.],
          [ 5.,  6.,  7.,  8.],
          [ 9., 10., 11., 12.],
          [13., 14., 15., 16.]]]])
tensor([[0.1713, 0.2691, 0.1906, 0.1912, 0.1778]])
